# 04 — Phase 3: GQA + KIVI (2-bit KV Cache)

**技术**: KIVI — 2-bit 非对称量化 KV Cache

### KIVI 原理
- **K Cache**: per-channel 量化（沿 head_dim 方向统计 min/max）
- **V Cache**: per-token 量化（沿 seq_len 方向统计 min/max）
- **Residual window**: 最近 128 个 token 保持 FP16（保证质量）
- 理论压缩 **~5-8×**

### CUDA Backend vs Python Reference
- **KIVI CUDA kernel**: 真正的加速（fused dequant + attention）
  - 需要编译: `cd KIVI && pip install -e .`
  - 见 `scripts/setup_kivi_jetson.sh`
- **Pure Python**: 本文件的 fallback，功能正确但**无加速**

### 预期效果
- KV Cache 显存占用从 FP16 降低 5-8×
- **TPOT 降低**: decode 每步搬运的 KV 数据量减少
- **TTFT 基本不变**: prefill 阶段 KV 尚未量化
- **PPL 可能略升**: 2-bit 有损压缩 → 需要检查质量退化

⚠️ 在 10-12 GB 统一内存上，KIVI 让我们能处理**更长的医疗文本**。

In [ ]:
import sys, gc, torch
sys.path.insert(0, '..')

from src.metrics import (
    measure_generation, run_benchmark, find_oom_threshold,
    print_memory_budget, JETSON_USABLE_GB,
)
from src.kivi_wrapper import (
    create_kivi_cache, is_cuda_backend_available, get_backend_info,
)
from src.perplexity import compute_ppl_with_kv_cache
from src.dataset_utils import load_prompts

print(f"KIVI Backend: {get_backend_info()}")
if not is_cuda_backend_available():
    print("\n⚠️  KIVI CUDA 后端未编译。使用 Python 参考实现。")
    print("    真正的延迟收益需要 CUDA kernel!")
    print("    编译方法: 见 scripts/setup_kivi_jetson.sh")

In [ ]:

from src.jetson_utils import load_model_safe, aggressive_cleanup

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
FALLBACK_NAME = "Qwen/Qwen2.5-0.5B-Instruct"
DEVICE = "cuda"

model, tokenizer = load_model_safe(
    MODEL_NAME,
    fallback_name=FALLBACK_NAME,
    device=DEVICE,
)
budget = print_memory_budget(model, DEVICE)


In [ ]:
# KIVI 参数
BITS = 2
GROUP_SIZE = 32
RESIDUAL_LENGTH = 128

prompts = load_prompts('../results/pubmedqa_prompts.json')
print(f"Loaded {len(prompts)} prompts")
print(f"\nKIVI config: bits={BITS}, group_size={GROUP_SIZE}, residual={RESIDUAL_LENGTH}")

---
## OOM 阈值（KIVI 应该能撑更长）

In [ ]:
print("KIVI OOM 探测...")
oom_kivi = find_oom_threshold(
    model, tokenizer,
    context_lengths=[256, 512, 1024, 2048, 4096, 8192, 16384],
    max_new_tokens=32,
    cache_factory=lambda: create_kivi_cache(
        residual_length=RESIDUAL_LENGTH,
        group_size=GROUP_SIZE,
        bits=BITS,
    ),
)

print(f"\n最大安全长度: {oom_kivi['max_safe_length']}")
print(f"OOM 发生在  : {oom_kivi['oom_length']}")
for r in oom_kivi['results']:
    if r['status'] == 'ok':
        print(f"  ctx={r['context_length']:>6}  peak={r['peak_memory_mb']:>7.0f} MB  "
              f"util={r['utilization']*100:>5.1f}%")
    else:
        print(f"  ctx={r['context_length']:>6}  → {r['status']}")

In [ ]:
# 单样本快速测试
cache = create_kivi_cache(RESIDUAL_LENGTH, GROUP_SIZE, BITS)
m = measure_generation(model, tokenizer, prompts[0]['prompt'],
                       max_new_tokens=64, cache_impl=cache)

print(f"TTFT           : {m.ttft_ms:.1f} ms")
print(f"TPOT           : {m.tpot_ms:.1f} ms")
print(f"KV Cache (est) : {m.kv_cache_memory_mb:.1f} MB")
print(f"Peak memory    : {m.peak_memory_mb:.0f} MB")
print(f"Memory util    : {m.memory_utilization*100:.1f}%")
print(f"\nCache info: {cache}")

In [ ]:
# 完整 benchmark
results_kivi = run_benchmark(
    model, tokenizer, prompts,
    cache_factory=lambda: create_kivi_cache(RESIDUAL_LENGTH, GROUP_SIZE, BITS),
    max_new_tokens=256,
    warmup_runs=2, num_runs=3,
)
print(f"Completed {len(results_kivi)} samples")

In [ ]:
import pandas as pd

df = pd.DataFrame(results_kivi)
df['config'] = 'GQA+KIVI'
df.to_csv('../results/gqa_kivi.csv', index=False)

print(df[['ttft_ms', 'tpot_ms', 'peak_memory_mb',
           'kv_cache_memory_mb', 'memory_utilization']].describe().round(1))

---
## PPL 退化评估

2-bit 量化可能导致生成质量下降。PPL 增加 > 5% 视为显著退化。

In [ ]:
# 用 KIVI cache 跑 PPL
eval_texts = [p['reference_answer'] for p in prompts[:20] if p.get('reference_answer')]

ppl_kivi = compute_ppl_with_kv_cache(
    model, tokenizer, eval_texts,
    cache_factory=lambda: create_kivi_cache(RESIDUAL_LENGTH, GROUP_SIZE, BITS),
    max_length=512,
)

# 加载 baseline PPL
import json
with open('../results/ppl_results.json') as f:
    ppl_all = json.load(f)

baseline_ppl = ppl_all['GQA_only']['ppl']
kivi_ppl = ppl_kivi['ppl']
degradation = (kivi_ppl - baseline_ppl) / baseline_ppl * 100

print(f"Baseline PPL : {baseline_ppl:.2f}")
print(f"KIVI 2-bit PPL: {kivi_ppl:.2f}")
print(f"退化         : {degradation:+.1f}%")
if abs(degradation) < 5:
    print("✓ 2-bit 量化质量可接受")
else:
    print("⚠️  PPL 退化超过 5%，可能影响医疗推理质量")

# 保存
ppl_all['GQA+KIVI'] = ppl_kivi
with open('../results/ppl_results.json', 'w') as f:
    json.dump(ppl_all, f, indent=2)

In [ ]:
# 生成质量抽检
print("=== KIVI 2-bit 生成示例 ===")
for i in [0, 10, 20]:
    if i < len(results_kivi):
        print(f"\n--- Sample {i}: {results_kivi[i].get('question', '')[:60]}... ---")
        print(results_kivi[i].get('sample_text', '')[:300])

In [ ]:
del model
gc.collect()
torch.cuda.empty_cache()
print(f"GPU memory after cleanup: {torch.cuda.memory_allocated() / (1024**2):.0f} MB")